![Image](./snapshot_strategy.png)

In [1]:
file_pre = '1st'

In [10]:
# import standard libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# import custom libraries
import lib.database.DBConn as db_conn
import lib.preprocess.preprocess as pp
import lib.preprocess.SectionExtractor as se

# import configurations of so)difficulty measure project
import setting_for_sdm.config as conf
from setting_for_sdm.constants import CONSTANTS

In [ ]:
htmlp = pp.HTMLParser()
codep = pp.CodeSectionParser()
ts = se.SectionExtractor()

In [4]:
db = db_conn.DBConn()

In [5]:
q_sql = """select a.id 
                , a.creationdate
                , a.title
                , b.body
            from posts a 
               , postsbody b
            where a.id = b.id
              and a.creationdate between %s and %s
              and a.posttypeid = '1'
              and a.tags like %s
              and not exists(select 1 
                                    from tt_posts_difficulty_annotated x 
                                where a.id = x.id)
              and not exists(select 1 
                                    from tt_posts_difficulty_done xx 
                                where a.id = xx.id)
"""

In [6]:
with db.cursor() as cur:
    cur.execute( q_sql,
                (CONSTANTS.start_date, CONSTANTS.end_date, '%<python>%')
                   )
    rows = cur.fetchall()  

In [7]:
q_output = pd.DataFrame(rows, columns = [
  'id'
  ,'creationdate'
  ,'title'
  , 'body'
])


In [8]:
print(q_output.creationdate.min())
print(q_output.creationdate.max())

2022-11-30 00:03:46.690000
2024-11-30 23:59:32.580000


In [13]:
q_output['bf'] = np.where(pd.to_datetime(q_output['creationdate']).dt.date < \
                             datetime.strptime(CONSTANTS.std_date, "%Y-%m-%d").date(), 1, 0)

qid_for_b = np.random.choice(q_output.loc[q_output['bf'] == 1, 'id'].tolist(), size=54, replace=False)
qid_for_f = np.random.choice(q_output.loc[q_output['bf'] == 0, 'id'].tolist(), size=54, replace=False)

In [ ]:
def chg_tag(code):
    st_pattern = r'<pre(?: class="[^"]*")?><code>'
    st_dst = "```python\n"
    code = re.sub(st_pattern, st_dst, code, count=0, flags=0)
    
    end_dst = "```"
    end_pattern =r'</code></pre>'
    code = re.sub(end_pattern, end_dst, code, count=0, flags=0)
    return code
                                                

In [ ]:
def clean_code_df(q_output, q_list):
    q_output_copy = q_output[q_output['id'].isin(q_list)].copy()
    q_output_copy['t_body'] = q_output_copy['body'].apply(chg_tag)
    q_output_copy['clean_body'] = q_output_copy['t_body'].apply(lambda x : htmlp.get_html_cleaned_str(x))
    q_output_copy['clean_body'] = q_output_copy['clean_body'].apply(lambda x:  re.sub(r";(?=\S)", "", x))
    q_output_copy['question'] = """<Title>"""+q_output_copy['title'].map(str)+"""</Title>. <Question>"""+q_output_copy['clean_body'].map(str)+"""</Question> Let's think through the difficulty of question carefully, step by step. """
    return q_output_copy


In [ ]:
pp_2324_22 = clean_code_df(q_output_22, qid_for_2223_22)
pp_2324_22 = clean_code_df(q_output_23, qid_for_2223_23)

pp_2324_22 = clean_code_df(q_output_for_2324_22, qid_for_2324_22)
pp_2324_23 = clean_code_df(q_output_for_2324_23, qid_for_2324_23)
pp_2324_24 = clean_code_df(q_output_for_2324_24, qid_for_2324_24)

In [ ]:
snapshop1_sample = pd.concat((pp_2324_22[['id', 'question']], pp_2324_22[['id', 'question']]))
snapshop2_sample = pd.concat((pp_2324_22[['id', 'question']], pp_2324_23[['id', 'question']], pp_2324_24[['id', 'question']]))

In [ ]:
snapshop1_sample[['id', 'question']].to_csv(f'./{file_pre}_snapshop1_sample.csv')
snapshop2_sample[['id', 'question']].to_csv(f'./{file_pre}_snapshop2_sample.csv')